# 03 — Model Training

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay, RocCurveDisplay
import joblib
import matplotlib.pyplot as plt

MODEL_DATA_PATH = Path('../data/modelling/model_data.csv')

model_data = pd.read_csv(MODEL_DATA_PATH)

y = model_data['home_win'].astype(int)
X = model_data.select_dtypes(include='number').drop(columns=['home_win'], errors='ignore')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Logistic Regression
log_reg = LogisticRegression(max_iter=500)
log_reg.fit(X_train, y_train)
log_pred = log_reg.predict(X_test)
print("LogReg Accuracy:", round(accuracy_score(y_test, log_pred), 4))
print(classification_report(y_test, log_pred))

# Random Forest
rf = RandomForestClassifier(n_estimators=400, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
print("RF Accuracy:", round(accuracy_score(y_test, rf_pred), 4))
print(classification_report(y_test, rf_pred))

# Plots
ConfusionMatrixDisplay.from_estimator(rf, X_test, y_test, cmap='Blues', values_format='d')
plt.title('Confusion Matrix — Random Forest (NBA-only)')
plt.show()

RocCurveDisplay.from_estimator(rf, X_test, y_test)
plt.title('ROC Curve — Random Forest (NBA-only)')
plt.show()

# Save model + feature order
Path('../models').mkdir(parents=True, exist_ok=True)
joblib.dump(rf, '../models/random_forest.pkl')
joblib.dump(list(X.columns), '../models/model_features.pkl')
print('Saved model + features')
